# 02 - Walk-forward model and backtest

Fits the configured model on purged walk-forward folds, converts the out-of-sample
scores into a cost-aware portfolio, and writes the predictions back to Delta.


In [ ]:
%pip install -e . yfinance lightgbm


In [ ]:
import matplotlib.pyplot as plt

from ml_trading.config import ExperimentConfig
from ml_trading.data import load_prices
from ml_trading.pipeline import run_experiment
from ml_trading.spark_io import PREDICTIONS_TABLE, write_table
from ml_trading.tracking import log_experiment

config = ExperimentConfig.from_yaml("configs/logistic_h21.yaml")
prices = load_prices(config.data)
result = run_experiment(config, prices)
print(result.report())


In [ ]:
daily = result.backtest.daily
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, height_ratios=[3, 1])
daily[["strategy_equity", "benchmark_equity"]].plot(ax=axes[0], logy=True)
axes[0].set_title(f"{config.name}: out-of-sample equity (net of {config.backtest.cost_bps}bps)")
daily["turnover"].rolling(21).mean().plot(ax=axes[1])
axes[1].set_ylabel("turnover (21d avg)")
plt.tight_layout()


In [ ]:
import pandas as pd

folds = pd.DataFrame(
    [
        {"fold": f.label, "train_rows": f.train_rows, "auc": f.auc, "ic": f.information_coefficient}
        for f in result.folds
    ]
)
display(folds)


In [ ]:
write_table(spark, result.predictions, PREDICTIONS_TABLE, partition_by="symbol")
log_experiment(result, experiment_name="/Shared/ml_trading")


### Reading the result honestly

The strategy line is *net* of transaction costs and is only ever earned from
`execution_lag` days after the signal. Compare it against `benchmark_sharpe` (the
equal-weight buy-and-hold book) and against `strategy_sharpe_pvalue`: with a decade
of daily data, a Sharpe below roughly 0.4 is not distinguishable from zero, and once
several configurations have been tried the bar is higher still - see
`deflated_sharpe_ratio` in `ml_trading.metrics`.
